# Local ID + Linear Neighbourhood Mapping as a Comparison Filter for Embedding Models

## Motivation

Given a fixed dataset, we want to know which of two embedding models (A vs B) produces better representations — **without any labels**. This notebook implements a geometric compatibility test:

1. Embed the same dataset with two models → clouds A and B (known sample correspondence).
2. Compute **per-sample local Intrinsic Dimensionality** for both clouds.
3. For each pair (i_A, i_B) with `lid_A[i] > lid_B[i]`, fit a **weighted least-squares linear map** from the k-NN neighbourhood of i in A to the corresponding points in B.  
   The system is **overdetermined** (k > d_B) — this is critical: underdetermined systems fit perfectly and carry no signal.
4. Collect the **mapping reconstruction errors** per sample.
5. Filter samples with large error — their local geometries are incompatible, meaning A and B cannot be meaningfully compared at those points.
6. Compare aggregate local ID of surviving samples to downstream retrieval quality for both models. The hypothesis: post-filter local ID correlates more strongly with downstream quality than the raw (all-sample) version.

### Formal Sketch

Let \( \mathcal{N}_k^A(i) = \{j_1, \ldots, j_k\} \) be the k-nearest neighbours of point \(x_i^A\) in cloud A, and let \( \Delta X^A_i \in \mathbb{R}^{k \times d_A} \) be the centred neighbour matrix (rows = \(x_{j_l}^A - x_i^A\)).  
Similarly \( \Delta X^B_i \in \mathbb{R}^{k \times d_B} \) uses the **same index set** from cloud B.

We solve the weighted overdetermined system:

\[
W_i = \underset{W}{\arg\min}\ \| W_i (\Delta X^A_i)^T - (\Delta X^B_i)^T \|^2_F \quad \text{s.t. } k > d_B
\]

with diagonal weight matrix \( \mathrm{diag}(w_1, \ldots, w_k) \) where

\[
w_l = \exp\!\left(-\frac{\|x_{j_l}^A - x_i^A\|^2}{2\sigma_i^2}\right), \quad \sigma_i = \text{mean kNN distance in } \mathcal{N}_k^A(i)
\]

The per-sample error is the normalised Frobenius residual:

\[
e_i = \frac{\| W_i (\Delta X^A_i)^T - (\Delta X^B_i)^T \|_F}{k \cdot d_B}
\]

Samples where \(e_i\) exceeds a chosen threshold \(\tau\) are flagged as **geometrically incompatible** and excluded from the comparison.


## 0. Install Dependencies

In [ ]:
# !pip install -q beir sentence-transformers skdim faiss-cpu pytrec_eval-terrier tqdm matplotlib scipy scikit-learn

## 1. Imports & Config

In [ ]:
import os
import random
import logging
import pathlib
from dataclasses import dataclass, field

import numpy as np
import pandas as pd
import skdim
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr
from sklearn.linear_model import LinearRegression
from tqdm.auto import tqdm

from beir import util
from beir.datasets.data_loader import GenericDataLoader
from beir.retrieval import models as beir_models
from beir.retrieval.evaluation import EvaluateRetrieval
from beir.retrieval.search.dense import DenseRetrievalExactSearch as DRES
from sentence_transformers import SentenceTransformer

logging.basicConfig(level=logging.WARNING)


@dataclass
class Config:
    # --- Dataset ---
    dataset: str = "nfcorpus"       # fixed dataset; use the same split for both models
    split: str = "test"
    max_queries: int = 323           # NFCorpus test = 323 queries, keep all
    max_corpus: int = 3633           # NFCorpus corpus = ~3.6k docs, keep all
    data_dir: str = "datasets"

    # --- Models to compare ---
    model_a: str = "sentence-transformers/all-MiniLM-L6-v2"   # baseline
    model_b: str = "sentence-transformers/all-mpnet-base-v2"   # stronger general model

    batch_size: int = 128

    # --- Local ID ---
    n_neighbors_id: int = 20         # k for local ID estimation

    # --- Linear mapping ---
    n_neighbors_map: int = 40        # k for neighbourhood used in linear map fitting
    # k must be > embedding dim for overdetermined system; 40 >> 384/768
    # Soft weights: Gaussian with adaptive sigma = mean kNN dist
    error_threshold_quantile: float = 0.75   # tau: filter top-25% highest-error samples

    # --- Downstream eval ---
    k_values: list = field(default_factory=lambda: [1, 3, 5, 10])


cfg = Config()
pathlib.Path(cfg.data_dir).mkdir(exist_ok=True)
random.seed(42)
np.random.seed(42)
print(cfg)

## 2. Load Dataset

In [ ]:
def load_beir_dataset(name: str, split: str = "test", data_dir: str = "datasets"):
    dataset_path = os.path.join(data_dir, name)
    if not os.path.exists(dataset_path):
        url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{name}.zip"
        util.download_and_unzip(url, data_dir)
    corpus, queries, qrels = GenericDataLoader(data_folder=dataset_path).load(split=split)
    return corpus, queries, qrels


corpus, queries, qrels = load_beir_dataset(cfg.dataset, split=cfg.split, data_dir=cfg.data_dir)
print(f"Dataset: {cfg.dataset}  |  {len(corpus):,} docs  |  {len(queries):,} queries")

# Sub-sample if needed
rng = random.Random(42)
valid_qids = [q for q in queries if q in qrels and qrels[q]]
sampled_qids = rng.sample(valid_qids, min(cfg.max_queries, len(valid_qids)))
queries_s = {q: queries[q] for q in sampled_qids}
qrels_s   = {q: qrels[q]   for q in sampled_qids}

relevant_cids = set(did for rels in qrels_s.values() for did in rels)
extra_cids = [c for c in corpus if c not in relevant_cids]
rng.shuffle(extra_cids)
keep_cids = list(relevant_cids) + extra_cids[:max(0, cfg.max_corpus - len(relevant_cids))]
corpus_s = {c: corpus[c] for c in keep_cids if c in corpus}

q_ids   = list(queries_s.keys())
q_texts = [queries_s[q] for q in q_ids]
print(f"Queries kept: {len(q_ids)}  |  Corpus docs kept: {len(corpus_s):,}")

## 3. Embed Queries with Both Models

We embed the **same query set** with both models to get two point clouds A and B
with known per-sample correspondence (index i in A ↔ index i in B).

In [ ]:
def embed(model_name_or_model, texts, batch_size=128):
    if isinstance(model_name_or_model, str):
        model = SentenceTransformer(model_name_or_model)
    else:
        model = model_name_or_model
    embs = model.encode(
        texts, batch_size=batch_size, show_progress_bar=True,
        normalize_embeddings=True, convert_to_numpy=True,
    )
    return embs


print(f"Embedding {len(q_texts)} queries with Model A: {cfg.model_a}")
emb_A = embed(cfg.model_a, q_texts, cfg.batch_size)

print(f"\nEmbedding {len(q_texts)} queries with Model B: {cfg.model_b}")
emb_B = embed(cfg.model_b, q_texts, cfg.batch_size)

print(f"\nCloud A: {emb_A.shape}  |  Cloud B: {emb_B.shape}")
assert emb_A.shape[0] == emb_B.shape[0], "Point clouds must have same number of points!"
N = emb_A.shape[0]

## 4. Per-Sample Local Intrinsic Dimensionality

Each cloud gets its own per-sample local ID computed **within that cloud's geometry**.
We use MLE (fast, reliable for moderate k).

Expected: the better-adapted model should show higher average local ID for this OOD dataset
(richer neighbourhood structure = more discriminative representation).

In [ ]:
def local_id_per_sample(emb: np.ndarray, n_neighbors: int = 20, estimator: str = "MLE") -> np.ndarray:
    """Return per-sample local intrinsic dimension array."""
    est = getattr(skdim.id, estimator)()
    dims = est.fit_transform_pw(emb.astype(np.float32), n_neighbors=n_neighbors, n_jobs=-1)
    return dims


print(f"Computing local ID for cloud A (k={cfg.n_neighbors_id}) ...")
lid_A = local_id_per_sample(emb_A, cfg.n_neighbors_id)

print(f"Computing local ID for cloud B (k={cfg.n_neighbors_id}) ...")
lid_B = local_id_per_sample(emb_B, cfg.n_neighbors_id)

print(f"\nLocal ID summary:")
print(f"  A ({cfg.model_a.split('/')[-1]}): mean={lid_A.mean():.2f}, median={np.median(lid_A):.2f}, std={lid_A.std():.2f}")
print(f"  B ({cfg.model_b.split('/')[-1]}): mean={lid_B.mean():.2f}, median={np.median(lid_B):.2f}, std={lid_B.std():.2f}")

## 5. Downstream Retrieval Quality for Both Models

We evaluate both models on the same dataset with BEIR to get the ground-truth quality
signal we want our unsupervised local ID metric to predict.

In [ ]:
def evaluate_retrieval(corpus, queries, qrels, model, batch_size=128, k_values=[1,3,5,10]):
    beir_model = DRES(beir_models.SentenceBERT(model), batch_size=batch_size)
    retriever  = EvaluateRetrieval(beir_model, score_function="dot", k_values=k_values)
    results    = retriever.retrieve(corpus, queries)
    ndcg, _map, recall, prec = retriever.evaluate(qrels, results, k_values=k_values)
    return ndcg, results


print("=== Evaluating Model A ===")
ndcg_A, results_A = evaluate_retrieval(corpus_s, queries_s, qrels_s, cfg.model_a, cfg.batch_size)

print("\n=== Evaluating Model B ===")
ndcg_B, results_B = evaluate_retrieval(corpus_s, queries_s, qrels_s, cfg.model_b, cfg.batch_size)

print(f"\nnDCG@10 — A: {ndcg_A['NDCG@10']:.4f}  |  B: {ndcg_B['NDCG@10']:.4f}")
print(f"Better model (nDCG@10): {'A' if ndcg_A['NDCG@10'] > ndcg_B['NDCG@10'] else 'B'}")
print(f"Predicted by higher mean local ID: {'A' if lid_A.mean() > lid_B.mean() else 'B'}")

## 6. Weighted Overdetermined Linear Mapping Between Neighbourhoods

For each sample i where `lid_A[i] > lid_B[i]`, we fit a linear map  
\( W_i: \mathbb{R}^{d_A} \to \mathbb{R}^{d_B} \) on the k-NN neighbourhood.

**Why overdetermined is essential**: if `k < d_A` (underdetermined), the system has an exact solution and the residual is zero regardless of how incompatible the geometries are — this is useless as a signal. We ensure `k >> max(d_A, d_B)` so the system is always overdetermined.

**Soft weights**: Gaussian kernel with adaptive per-sample bandwidth  
\(\sigma_i = \text{mean}_{j \in \mathcal{N}_k(i)} \| x_i - x_j \|\), so nearer neighbours have more influence — avoiding the crude hard-boundary of a fixed ball while keeping locality.

**Normalised residual** per sample:  
\[ e_i = \frac{\|W_i \, \Delta X_i^{A\top} - \Delta X_i^{B\top}\|_F}{k \cdot d_B} \]

In [ ]:
from numpy.linalg import lstsq

def compute_mapping_errors(
    emb_src: np.ndarray,   # higher-dim cloud (map FROM here)
    emb_tgt: np.ndarray,   # lower-dim cloud (map TO here)
    k: int = 40,
    use_weights: bool = True,
) -> np.ndarray:
    """
    For each point i, find k-NN in emb_src, get corresponding points in emb_tgt
    (by index — known correspondence), fit weighted overdetermined linear map,
    return normalised residual per sample.

    System size: k equations, d_src unknowns per output dim.
    Overdetermined requires k > d_src (always true here since k=40, d<=768).
    """
    N, d_src = emb_src.shape
    d_tgt = emb_tgt.shape[1]

    # Build kNN index for emb_src
    from sklearn.neighbors import NearestNeighbors
    nn = NearestNeighbors(n_neighbors=k + 1, metric="euclidean", algorithm="auto", n_jobs=-1)
    nn.fit(emb_src)
    dists, idxs = nn.kneighbors(emb_src)  # includes self at pos 0
    # drop self (index 0)
    dists = dists[:, 1:]   # (N, k)
    idxs  = idxs[:, 1:]    # (N, k)

    errors = np.zeros(N, dtype=np.float64)

    for i in range(N):
        nbr_idx = idxs[i]          # shape (k,)
        d_nbr   = dists[i]         # shape (k,)

        # Centred neighbour matrices
        dx_src = emb_src[nbr_idx] - emb_src[i]   # (k, d_src)
        dx_tgt = emb_tgt[nbr_idx] - emb_tgt[i]   # (k, d_tgt)

        # Adaptive Gaussian weights
        if use_weights:
            sigma  = d_nbr.mean() + 1e-12
            w      = np.exp(-d_nbr**2 / (2 * sigma**2))
            w      = w / (w.sum() + 1e-12)
            sqrt_w = np.sqrt(w)[:, None]           # (k, 1) broadcast
            A_mat  = dx_src * sqrt_w               # (k, d_src)
            B_mat  = dx_tgt * sqrt_w               # (k, d_tgt)
        else:
            A_mat = dx_src
            B_mat = dx_tgt

        # Solve overdetermined system A_mat @ W.T = B_mat  → W.T: (d_src, d_tgt)
        # lstsq solves min ||A_mat @ X - B_mat||_F
        W_T, residuals, rank, sv = lstsq(A_mat, B_mat, rcond=None)

        # Compute residual manually (lstsq residuals only returned when k > d_src)
        B_hat  = A_mat @ W_T
        frob   = np.linalg.norm(B_hat - B_mat, "fro")
        errors[i] = frob / (k * d_tgt)

    return errors


print(f"k_map={cfg.n_neighbors_map}, d_A={emb_A.shape[1]}, d_B={emb_B.shape[1]}")
print(f"System is overdetermined: k({cfg.n_neighbors_map}) > max(d_A, d_B)({max(emb_A.shape[1], emb_B.shape[1])})")
print()

# Determine mapping direction per sample: map FROM higher-lid cloud
# For samples where lid_A > lid_B: map A → B
# For samples where lid_B > lid_A: map B → A
# This respects the surjection intuition (higher-dim → lower-dim is more constrained)
direction = (lid_A > lid_B).astype(int)   # 1 = A→B, 0 = B→A
n_A2B = direction.sum()
n_B2A = N - n_A2B
print(f"Direction: {n_A2B} samples map A→B, {n_B2A} samples map B→A")

# Compute errors in each direction
print("\nComputing A→B mapping errors ...")
err_A2B = compute_mapping_errors(emb_A, emb_B, k=cfg.n_neighbors_map)

print("Computing B→A mapping errors ...")
err_B2A = compute_mapping_errors(emb_B, emb_A, k=cfg.n_neighbors_map)

# Per-sample error: use direction-appropriate error
errors = np.where(direction == 1, err_A2B, err_B2A)

print(f"\nMapping error summary:")
print(f"  mean={errors.mean():.6f}, median={np.median(errors):.6f}, "
      f"std={errors.std():.6f}, max={errors.max():.6f}")

## 7. Visualise Mapping Error Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Error histogram
axes[0].hist(errors, bins=40, edgecolor="none", alpha=0.7, color="#5C6BC0")
tau = np.percentile(errors, cfg.error_threshold_quantile * 100)
axes[0].axvline(tau, color="red", lw=1.5, ls="--",
                label=f"τ = {tau:.4f} ({cfg.error_threshold_quantile*100:.0f}th pct)")
axes[0].set_xlabel("Normalised mapping error")
axes[0].set_ylabel("Count")
axes[0].set_title("Distribution of Neighbourhood Mapping Errors")
axes[0].legend()

# Error vs |lid_A - lid_B|
lid_diff = np.abs(lid_A - lid_B)
axes[1].scatter(lid_diff, errors, alpha=0.3, s=12, color="#26A69A")
axes[1].set_xlabel("|local_id_A - local_id_B|")
axes[1].set_ylabel("Mapping error")
axes[1].set_title("Mapping Error vs Local ID Discrepancy")
# Add correlation
r, p = spearmanr(lid_diff, errors)
axes[1].text(0.02, 0.95, f"Spearman r={r:.3f}, p={p:.2e}",
             transform=axes[1].transAxes, va="top", fontsize=9)

plt.tight_layout()
plt.savefig("mapping_errors.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"Threshold τ (quantile {cfg.error_threshold_quantile}): {tau:.6f}")

## 8. Filter Incompatible Samples & Compare Local ID to Quality

After removing high-error samples (geometrically incompatible neighbourhoods), we compare:
- **Aggregate local ID** (mean and median) of each model's cloud for surviving samples
- Whether the model with higher aggregate local ID is the one with higher nDCG@10
- Whether the signal is cleaner (larger separation) after filtering

In [ ]:
tau = np.percentile(errors, cfg.error_threshold_quantile * 100)
keep_mask = errors <= tau
keep_idx  = np.where(keep_mask)[0]

print(f"Threshold τ = {tau:.6f}")
print(f"Kept:    {keep_mask.sum()} / {N} samples ({keep_mask.mean()*100:.1f}%)")
print(f"Removed: {(~keep_mask).sum()} samples")

# Aggregate local ID before and after filtering
stats = {}
for tag, mask in [("All samples", np.ones(N, bool)), ("Filtered (kept)", keep_mask)]:
    stats[tag] = {
        "n": int(mask.sum()),
        "lid_A_mean": lid_A[mask].mean(),
        "lid_A_median": np.median(lid_A[mask]),
        "lid_B_mean": lid_B[mask].mean(),
        "lid_B_median": np.median(lid_B[mask]),
        "lid_A_higher_mean": lid_A[mask].mean() > lid_B[mask].mean(),
        "lid_A_higher_median": np.median(lid_A[mask]) > np.median(lid_B[mask]),
    }

print(f"\n{'':30s} {'All':>12s}  {'Filtered':>12s}")
print("-" * 58)
for key in ["lid_A_mean", "lid_A_median", "lid_B_mean", "lid_B_median"]:
    print(f"{key:30s} {stats['All samples'][key]:12.4f}  {stats['Filtered (kept)'][key]:12.4f}")

print(f"\nGround truth: nDCG@10  A={ndcg_A['NDCG@10']:.4f}  B={ndcg_B['NDCG@10']:.4f}")
gt_A_better = ndcg_A['NDCG@10'] > ndcg_B['NDCG@10']
print(f"Better model by nDCG@10: {'A' if gt_A_better else 'B'}")
print()
for tag in ["All samples", "Filtered (kept)"]:
    pred_A_mean   = stats[tag]["lid_A_higher_mean"]
    pred_A_median = stats[tag]["lid_A_higher_median"]
    sep_mean   = abs(stats[tag]["lid_A_mean"]   - stats[tag]["lid_B_mean"])
    sep_median = abs(stats[tag]["lid_A_median"] - stats[tag]["lid_B_median"])
    correct_mean   = pred_A_mean   == gt_A_better
    correct_median = pred_A_median == gt_A_better
    print(f"[{tag}]")
    print(f"  Prediction by mean LID:   {'A' if pred_A_mean else 'B'}  (correct: {correct_mean})  separation: {sep_mean:.4f}")
    print(f"  Prediction by median LID: {'A' if pred_A_median else 'B'}  (correct: {correct_median})  separation: {sep_median:.4f}")

## 9. Sweep Mapping Error Threshold

We sweep the quantile threshold and track:
- Separation between `mean_lid_A` and `mean_lid_B` on surviving samples
- Whether the ordering agrees with ground-truth nDCG@10

The ideal threshold is the one where (a) the correct model is identified and (b) the LID separation is maximised.

In [ ]:
quantiles = np.arange(0.0, 1.0, 0.05)
sweep_rows = []

for q in quantiles:
    thr = np.percentile(errors, q * 100)
    mask = errors <= thr
    if mask.sum() < 5:
        continue
    lid_A_m = lid_A[mask].mean()
    lid_B_m = lid_B[mask].mean()
    lid_A_med = np.median(lid_A[mask])
    lid_B_med = np.median(lid_B[mask])
    sweep_rows.append({
        "quantile_kept": 1.0 - q,
        "n_kept": int(mask.sum()),
        "lid_A_mean": lid_A_m,
        "lid_B_mean": lid_B_m,
        "sep_mean": abs(lid_A_m - lid_B_m),
        "pred_mean_correct": (lid_A_m > lid_B_m) == gt_A_better,
        "lid_A_median": lid_A_med,
        "lid_B_median": lid_B_med,
        "sep_median": abs(lid_A_med - lid_B_med),
        "pred_median_correct": (lid_A_med > lid_B_med) == gt_A_better,
    })

sweep_df = pd.DataFrame(sweep_rows)
print(sweep_df[["quantile_kept", "n_kept", "lid_A_mean", "lid_B_mean",
                "sep_mean", "pred_mean_correct"]].to_string(index=False))
sweep_df.to_csv("threshold_sweep_mapping.csv", index=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Separation vs fraction kept
ax = axes[0]
ax.plot(sweep_df["quantile_kept"] * 100, sweep_df["sep_mean"],   marker="o", label="Mean LID sep.")
ax.plot(sweep_df["quantile_kept"] * 100, sweep_df["sep_median"], marker="s", ls="--", label="Median LID sep.")
# Mark where prediction is correct
correct_mean = sweep_df[sweep_df["pred_mean_correct"]]
ax.scatter(correct_mean["quantile_kept"]*100, correct_mean["sep_mean"],
           marker="*", s=120, color="green", zorder=5, label="Correct prediction")
ax.invert_xaxis()
ax.set_xlabel("% samples kept (left = more filtering)")
ax.set_ylabel("|LID_A - LID_B|")
ax.set_title("LID Separation vs Filter Aggressiveness")
ax.legend(fontsize=8)

# Per-sample: lid_A vs lid_B scatter coloured by mapping error
ax2 = axes[1]
sc = ax2.scatter(lid_A, lid_B, c=errors, cmap="RdYlGn_r", s=12, alpha=0.6)
plt.colorbar(sc, ax=ax2, label="Mapping error")
mn = min(lid_A.min(), lid_B.min())
mx = max(lid_A.max(), lid_B.max())
ax2.plot([mn, mx], [mn, mx], "k--", lw=0.8, alpha=0.4, label="lid_A = lid_B")
ax2.set_xlabel(f"Local ID — Model A ({cfg.model_a.split('/')[-1]})")
ax2.set_ylabel(f"Local ID — Model B ({cfg.model_b.split('/')[-1]})")
ax2.set_title("Per-Sample Local ID Comparison\n(colour = mapping error)")
ax2.legend(fontsize=8)

plt.tight_layout()
plt.savefig("lid_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 10. Per-Query nDCG@10 Correlation with Local ID

If local ID is a useful proxy, per-query local ID should correlate with per-query retrieval success.
We compute this correlation before and after filtering for both models.

In [ ]:
def per_query_ndcg(results, qrels, k=10):
    """Compute per-query nDCG@k."""
    from beir.retrieval.evaluation import EvaluateRetrieval
    # EvaluateRetrieval doesn't expose per-query nDCG directly, so we compute manually.
    import math
    ndcg_per_q = {}
    for qid, doc_scores in results.items():
        if qid not in qrels:
            continue
        rel = qrels[qid]
        ranked = sorted(doc_scores.items(), key=lambda x: -x[1])[:k]
        dcg  = sum(rel.get(did, 0) / math.log2(r + 2) for r, (did, _) in enumerate(ranked))
        # ideal DCG
        ideal = sorted(rel.values(), reverse=True)[:k]
        idcg = sum(v / math.log2(r + 2) for r, v in enumerate(ideal)) if ideal else 1.0
        ndcg_per_q[qid] = dcg / idcg if idcg > 0 else 0.0
    return ndcg_per_q


ndcg10_A_pq = per_query_ndcg(results_A, qrels_s, k=10)
ndcg10_B_pq = per_query_ndcg(results_B, qrels_s, k=10)

# Align with q_ids order
ndcg_A_arr = np.array([ndcg10_A_pq.get(q, np.nan) for q in q_ids])
ndcg_B_arr = np.array([ndcg10_B_pq.get(q, np.nan) for q in q_ids])

print("Spearman correlation: local ID vs per-query nDCG@10")
print()

for name, lid, ndcg_arr in [("A", lid_A, ndcg_A_arr), ("B", lid_B, ndcg_B_arr)]:
    for tag, mask in [("all", np.ones(N, bool)), ("filtered", keep_mask)]:
        valid = ~np.isnan(ndcg_arr) & mask
        if valid.sum() < 3:
            continue
        r, p = spearmanr(lid[valid], ndcg_arr[valid])
        print(f"  Model {name} [{tag:8s}]  n={valid.sum():3d}  Spearman r={r:+.4f}  p={p:.3e}")

print()
print("Prediction of better model by mean local ID:")
print(f"  All samples :  A_mean={lid_A.mean():.3f}  B_mean={lid_B.mean():.3f}  → {'A' if lid_A.mean()>lid_B.mean() else 'B'}")
print(f"  Filtered    :  A_mean={lid_A[keep_mask].mean():.3f}  B_mean={lid_B[keep_mask].mean():.3f}  → {'A' if lid_A[keep_mask].mean()>lid_B[keep_mask].mean() else 'B'}")
print(f"  Ground truth (nDCG@10): {'A' if gt_A_better else 'B'} is better")

## 11. Per-Query Local ID vs nDCG@10

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for row, (name, lid, ndcg_arr, results) in enumerate([
    ("A", lid_A, ndcg_A_arr, results_A),
    ("B", lid_B, ndcg_B_arr, results_B),
]):
    for col, (tag, mask) in enumerate([("All", np.ones(N, bool)), ("Filtered", keep_mask)]):
        ax = axes[row][col]
        valid = ~np.isnan(ndcg_arr) & mask
        sc = ax.scatter(lid[valid], ndcg_arr[valid], c=errors[valid],
                        cmap="RdYlGn_r", s=15, alpha=0.5, vmin=0, vmax=np.percentile(errors, 95))
        r, p = spearmanr(lid[valid], ndcg_arr[valid])
        ax.set_title(f"Model {name} | {tag}  (n={valid.sum()})\nSpearman r={r:.3f}, p={p:.2e}")
        ax.set_xlabel(f"Local ID (Model {name})")
        ax.set_ylabel("nDCG@10")
        plt.colorbar(sc, ax=ax, label="map error")

plt.suptitle("Per-Query: Local ID vs nDCG@10 (colour = mapping error)", y=1.01)
plt.tight_layout()
plt.savefig("local_id_vs_ndcg_perquery.png", dpi=150, bbox_inches="tight")
plt.show()

## 12. Alternative Threshold Strategies for τ

We discuss three principled ways to set τ (rather than a fixed quantile):

1. **Quantile-based** (default above): simple, no assumptions.
2. **Residual variance test**: under the null that the map is a true linear isometry, the expected normalised error scales as \(\sigma_{noise} / \sqrt{k}\). Estimate \(\hat{\sigma}\) from the median error and set \(\tau = 3\hat{\sigma}/\sqrt{k}\) (3-sigma rule).
3. **Elbow method on sorted errors**: fit two line segments to the sorted error curve, find the inflection point.

In [ ]:
# Strategy 2: 3-sigma noise estimate
sigma_hat = np.median(errors) / 0.6745   # robust sigma via median absolute deviation proxy
tau_noise  = 3 * sigma_hat / np.sqrt(cfg.n_neighbors_map)
n_kept_noise = (errors <= tau_noise).sum()
print(f"Strategy 2 (3σ noise): τ={tau_noise:.6f}  →  {n_kept_noise}/{N} kept ({n_kept_noise/N*100:.1f}%)")

# Strategy 3: elbow / kneedle on sorted errors
sorted_errors = np.sort(errors)
x = np.linspace(0, 1, N)
# Kneedle: normalise, find max deviation from diagonal
y = (sorted_errors - sorted_errors.min()) / (sorted_errors.max() - sorted_errors.min() + 1e-12)
diff = y - x
elbow_idx = np.argmax(diff)
tau_elbow = sorted_errors[elbow_idx]
n_kept_elbow = (errors <= tau_elbow).sum()
print(f"Strategy 3 (elbow)   : τ={tau_elbow:.6f}  →  {n_kept_elbow}/{N} kept ({n_kept_elbow/N*100:.1f}%)")

# Plot sorted errors with thresholds
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.arange(N), sorted_errors, color="#5C6BC0", lw=1.2, label="Sorted errors")
ax.axvline(elbow_idx, color="orange", ls="--", lw=1.5, label=f"Elbow τ={tau_elbow:.4f}")
ax.axhline(tau_noise, color="green",  ls=":",  lw=1.5, label=f"3σ-noise τ={tau_noise:.4f}")
ax.axhline(tau,       color="red",    ls="-.", lw=1.5, label=f"Quantile({cfg.error_threshold_quantile}) τ={tau:.4f}")
ax.set_xlabel("Sample rank (sorted by error)")
ax.set_ylabel("Normalised mapping error")
ax.set_title("Sorted Mapping Errors with Alternative Thresholds")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig("threshold_strategies.png", dpi=150, bbox_inches="tight")
plt.show()

# Compare strategies
print("\nComparison of threshold strategies:")
for strat_name, strat_tau in [
    (f"Quantile({cfg.error_threshold_quantile})", tau),
    ("3σ-noise", tau_noise),
    ("Elbow", tau_elbow),
]:
    m = errors <= strat_tau
    if m.sum() < 3:
        continue
    A_m, B_m = lid_A[m].mean(), lid_B[m].mean()
    correct = (A_m > B_m) == gt_A_better
    print(f"  {strat_name:20s}  n_kept={m.sum():3d}  A_mean={A_m:.3f}  B_mean={B_m:.3f}  "
          f"pred={'A' if A_m>B_m else 'B'}  correct={correct}")

## 13. Summary

In [ ]:
print("=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"Dataset:  {cfg.dataset}  ({N} queries)")
print(f"Model A:  {cfg.model_a}")
print(f"Model B:  {cfg.model_b}")
print()
print(f"Ground-truth nDCG@10:  A={ndcg_A['NDCG@10']:.4f}  B={ndcg_B['NDCG@10']:.4f}")
print(f"Better model: {'A' if gt_A_better else 'B'}")
print()
print(f"{'Metric':35s} {'Predicts A better':>18s}  {'Correct':>8s}")
print("-" * 65)

checks = [
    ("Raw mean local ID",  lid_A.mean() > lid_B.mean()),
    ("Raw median local ID", np.median(lid_A) > np.median(lid_B)),
    (f"Filtered mean LID (q={cfg.error_threshold_quantile})", lid_A[keep_mask].mean() > lid_B[keep_mask].mean()),
    (f"Filtered median LID (q={cfg.error_threshold_quantile})", np.median(lid_A[keep_mask]) > np.median(lid_B[keep_mask])),
]
for desc, pred_A in checks:
    correct = pred_A == gt_A_better
    print(f"{desc:35s} {'Yes' if pred_A else 'No':>18s}  {'✓' if correct else '✗':>8s}")

print()
print("Spearman (local ID vs per-query nDCG@10):")
for name, lid, ndcg_arr in [("A", lid_A, ndcg_A_arr), ("B", lid_B, ndcg_B_arr)]:
    for tag, mask in [("all", np.ones(N, bool)), ("filtered", keep_mask)]:
        valid = ~np.isnan(ndcg_arr) & mask
        if valid.sum() < 3:
            continue
        r, p = spearmanr(lid[valid], ndcg_arr[valid])
        print(f"  Model {name} [{tag:8s}]  r={r:+.4f}  p={p:.3e}")